In [1]:
from bs4 import BeautifulSoup
import aiohttp
from UMS.app.constants import constant
from UMS.app.schema.user_schema import UserLogin
from UMS.app import error_status
import json
from UMS.attendance_summary import get_attendance_summary, get_attendance_detail
from UMS.get_time_table import get_time_table_details
from datetime import datetime, timedelta

async def login_using_reg_no_ums_home(user: UserLogin):
    url = constant.UMS_LOGIN_URL
    headers = constant.USER_AGENT_FORM_URL_ENCODED
    async with aiohttp.ClientSession() as session:
        async with session.get(
            url,
            headers=headers,
        ) as res:
            #! This will get us first event and view state
            html = await res.text()
            soup = BeautifulSoup(html, "html.parser")
            __LASTFOCUS = ""
            __EVENTTARGET = ""
            __EVENTARGUMENT = ""
            __VIEWSTATE = soup.find("input", {"id": "__VIEWSTATE"})["value"]
            __VIEWSTATEGENERATOR = soup.find("input", {"id": "__VIEWSTATEGENERATOR"})[
                "value"
            ]
            __SCROLLPOSITIONX = "0"
            __SCROLLPOSITIONY = "0"
            __EVENTVALIDATION = soup.find("input", {"id": "__EVENTVALIDATION"})["value"]
            txtU = user.reg_no
            TxtpwdAutoId_8767 = user.password
            DropDownList1 = "1"
            ddlStartWith = "StudentDashboard.aspx"
            iBtnLogins_x = "40"
            iBtnLogins_y = "50"

            #! Payload just with reg_number.
            payload_with_reg_no_only = {
                "__LASTFOCUS": __LASTFOCUS,
                "__EVENTTARGET": "txtU",
                "__EVENTARGUMENT": __EVENTARGUMENT,
                "__VIEWSTATE": (__VIEWSTATE),
                "__VIEWSTATEGENERATOR": __VIEWSTATEGENERATOR,
                "__SCROLLPOSITIONX": __SCROLLPOSITIONX,
                "__SCROLLPOSITIONY": __SCROLLPOSITIONY,
                "__EVENTVALIDATION": (__EVENTVALIDATION),
                "txtU": txtU,
                "TxtpwdAutoId_8767": "",
                "DropDownList1": DropDownList1,
            }
            soup.decompose()
            #! Here we will make a post request just with the user id and it will give us updated states
            async with session.post(
                url, headers=headers, data=payload_with_reg_no_only
            ) as res:
                html = await res.text()
                soup = BeautifulSoup(html, "html.parser")

                __VIEWSTATE = soup.find("input", {"id": "__VIEWSTATE"})["value"]
                __VIEWSTATEGENERATOR = soup.find(
                    "input", {"id": "__VIEWSTATEGENERATOR"}
                )["value"]
                __SCROLLPOSITIONX = "0"
                __SCROLLPOSITIONY = "0"
                __EVENTVALIDATION = soup.find("input", {"id": "__EVENTVALIDATION"})[
                    "value"
                ]

                #! Payload with updated state and Password
                payload = {
                    "__LASTFOCUS": __LASTFOCUS,
                    "__EVENTTARGET": __EVENTTARGET,
                    "__EVENTARGUMENT": __EVENTARGUMENT,
                    "__VIEWSTATE": (__VIEWSTATE),
                    "__VIEWSTATEGENERATOR": __VIEWSTATEGENERATOR,
                    "__SCROLLPOSITIONX": __SCROLLPOSITIONX,
                    "__SCROLLPOSITIONY": __SCROLLPOSITIONY,
                    "__EVENTVALIDATION": (__EVENTVALIDATION),
                    "txtU": txtU,
                    "TxtpwdAutoId_8767": TxtpwdAutoId_8767,
                    "ddlStartWith": ddlStartWith,
                    "iBtnLogins150203125": "Login",
                    # "iBtnLogins.x": iBtnLogins_x,
                    # "iBtnLogins.y": iBtnLogins_y,
                }
                soup.decompose()
                #! SignIn with updated payload
                async with session.post(url, headers=headers, data=payload) as res:
                    asp_cookie = None
                    try:
                        asp_cookie = res.request_info.headers["Cookie"]
                    except:
                        await session.close()
                        raise error_status.SOMETHING_WRONG_WITH_UMS_SERVER
                    await session.close()
                    is_auth = await check_auth_status_ums_home(asp_cookie)
                    if is_auth:
                        return asp_cookie
                    raise error_status.CREDENTIALS_NOT_VALID

async def check_auth_status_ums_home(cookie) -> bool:
    async with aiohttp.ClientSession() as session:
        headers = constant.USER_AGENT_JSON
        headers["Cookie"] = cookie
        resp = await session.post(
            constant.UMS_STUDENT_PHONE_NUMBER_URL,
            headers=headers,
            data=json.dumps({}),
        )
        rs_json = await resp.json()
        await session.close()
        if rs_json.get("d", {}) is None:
            return False
        return True
    
with open('timetable.json', "r", encoding="utf-8") as file:
        timetable = json.load(file)

current_date = datetime.today().strftime('%m/%d/%Y')
old_date = timetable["last_updated"].split(" ")[0]
old_date = datetime.strptime(old_date, "%m/%d/%Y").date().strftime('%m/%d/%Y')

if(old_date != current_date):
        print("New Day!, Updating Details...")
        akhil = UserLogin(reg_no="12309014", password="@Ak6918g")
        cookies = await (login_using_reg_no_ums_home(akhil))
        timetable = await get_time_table_details(cookies)
        with open('timetable.json', 'w', encoding='utf-8') as f:
                json.dump(timetable, f, ensure_ascii=False, indent=4)

course_codes = {}
for key, value in timetable['faculty_details'].items():
    course_codes[key] = value['course_title']

In [2]:
current_day = datetime.now().strftime('%A')
todays_schedule = timetable["time_table"]["Tuesday"]
todays_schedule

{'09-10 AM': '',
 '10-11 AM': 'Lecture / G:All C:MTH302 / R: 28-507 / S:K23WT',
 '11-12 AM': 'Lecture / G:All C:CSE211 / R: 28-507 / S:K23WT',
 '12-01 PM': '',
 '01-02 PM': 'Lecture / G:All C:CSE310 / R: 28-505 / S:K23WT',
 '02-03 PM': 'Lecture / G:All C:CSE310 / R: 28-505 / S:K23WT',
 '03-04 PM': 'Lecture / G:All C:CSE408 / R: 28-505 / S:K23WT',
 '04-05 PM': ''}

In [53]:
import json
file_path = "timeTableContext.json"
with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)
print("Number of entries:", len(data))

Number of entries: 50


In [41]:
with open('timetable.json', "r", encoding="utf-8") as file:
        timetable = json.load(file)
        
def format_input(entry):
    instruction_text = (
        f"Given the query about today's schedule and the provided context, "
        f"generate a response that clearly conveys the scheduled activity in a natural and concise manner."
        f"\n\n### Query:\n{entry['query']}\n"
    )

    isfirst = True
    day = entry["current_time"].split(" ")[-1]
    todays_schedule = timetable["time_table"][day]
    now = datetime.now()
    today9am = now.replace(hour=9, minute=0, second=0, microsecond=0)
    today4pm = now.replace(hour=16, minute=0, second=0, microsecond=0)
    today430pm = now.replace(hour=16, minute=20, second=0, microsecond=0)
    
    time_table_dict = {}
    for key in todays_schedule:
        text = todays_schedule[key].replace("/","").split(" ")
        text = [x for x in text if x.strip()]
        start_hour, end_hour, am_pm = key[:2], key[3:5], key[6:]
        time_table_time = datetime.strptime(f"{start_hour} {am_pm}", "%I %p").replace(year=now.year, month=now.month, day=now.day)
        if(len(text) > 0):
            input_text = f"{text[0]} of {text[2].replace("C:","")} in Room {text[4]}\n"
        else:
            if(isfirst and time_table_time > today9am):
                input_text = f"Lunch break"
                isfirst = False
            else:
                input_text = f"No class right now"
        time_table_dict[f"{time_table_time.strftime('%H:%M')}"] = input_text

    result = "pp"
    curr_time = datetime.strptime(entry["current_time"], '%H:%M %A')
    if(curr_time.replace(year=now.year, month=now.month, day=now.day) > today4pm):
        if(curr_time.replace(year=now.year, month=now.month, day=now.day) > today430pm):
            result = "Classes are over for today."
        else:
            result = time_table_dict[f"16:00"]
    else:
        for time in time_table_dict:
            time_table_time = (datetime.strptime(time, '%H:%M') + timedelta(minutes=7))
            if(curr_time < time_table_time):
                lecture = time_table_dict[time].strip()
                if(lecture == "No class right now" and time != f"16:00"):
                    continue
                else:
                    if(lecture == "No class right now"):
                        result = f"{lecture}"
                    else:
                        result = f"{lecture} at {time} today."
                    break
    for key, value in course_codes.items():
        if(key in result):
            result = result.replace(key, value)
            break
    input_text = f"\n### Context:\n" + result
    return time_table_dict
    return instruction_text + input_text



In [45]:
def format_input(entry):
    instruction_text = (
        f"Given the query about today's schedule and the provided context, "
        f"generate a response that clearly conveys the scheduled activity in a natural and concise manner."
        f"\n\n### Query:\n{entry['query']}"
        f"\n\n### Current Time:\n{entry['current_time']}\n"
    )
    
    timetable = (
        f"Time Table Of Tuesday:\n"
        f"Tutorial of ANALYTICAL SKILLS-I in Room 34-704 is at 09:00\n"
        f"Practical of PROGRAMMING IN JAVA in Room 28-507A is at 10:00\n"
        f"Practical of PROGRAMMING IN JAVA in Room 28-507A is at 11:00\n"
        f"Lunch break is at 12:00\n"
        f"Lecture of SOFT COMPUTING in Room 28-507A is at 13:00\n"
        f"Lecture of SOFT COMPUTING in Room 28-507A is at 14:00\n"
        f"Lecture of ANALYTICAL SKILLS-I in Room 28-507A is at 15:00\n"
        f"Tutorial of PROBABILITY AND STATISTICS in Room 34-704 is at 16:00"
    )
    input_text = f"\n### Context:\n" + timetable

    return instruction_text + input_text

In [46]:
file_path = "tableReasoning.json"
with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

print(format_input(data[0]))

Given the query about today's schedule and the provided context, generate a response that clearly conveys the scheduled activity in a natural and concise manner.

### Query:
Which class do I have in the next hour?

### Current Time:
08:30 Monday

### Context:
Time Table Of Tuesday:
Tutorial of ANALYTICAL SKILLS-I in Room 34-704 is at 09:00
Practical of PROGRAMMING IN JAVA in Room 28-507A is at 10:00
Practical of PROGRAMMING IN JAVA in Room 28-507A is at 11:00
Lunch break is at 12:00
Lecture of SOFT COMPUTING in Room 28-507A is at 13:00
Lecture of SOFT COMPUTING in Room 28-507A is at 14:00
Lecture of ANALYTICAL SKILLS-I in Room 28-507A is at 15:00
Tutorial of PROBABILITY AND STATISTICS in Room 34-704 is at 16:00


In [43]:
entry ={ "query": "What's my schedule for today?", 
        "current_time": "12:00 Tuesday" 
    }
dd = format_input(entry)

for time, lecture in dd.items():
    for key, value in course_codes.items():
        if(key in lecture):
            dd[time] = lecture.replace(key, value)
            break

print("Time Table Of Tuesday")
for time, lecture in dd.items():
    print(time ," -> ", lecture.strip())

Time Table Of Tuesday
09:00  ->  Tutorial of ANALYTICAL SKILLS-I in Room 34-704
10:00  ->  Practical of PROGRAMMING IN JAVA in Room 28-507A
11:00  ->  Practical of PROGRAMMING IN JAVA in Room 28-507A
12:00  ->  Lunch break
13:00  ->  Lecture of SOFT COMPUTING in Room 28-507A
14:00  ->  Lecture of SOFT COMPUTING in Room 28-507A
15:00  ->  Lecture of ANALYTICAL SKILLS-I in Room 28-507A
16:00  ->  Tutorial of PROBABILITY AND STATISTICS in Room 34-704


In [5]:
formatted_data = {}
file_path = "timeTableContext.json"
with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

for entry in data:
    context = format_input(entry)
    formatted_data[context] = entry['current_time']


In [89]:
len(formatted_data)

31

In [6]:
file_path = "timeTableContext2.json"
with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)
print("Number of entries:", len(data))
resp = {}
for entry in data:
    resp[entry['context']] = entry['response']


Number of entries: 31


In [101]:
file_path = "timeTableContext.json"
with open(file_path, "r", encoding="utf-8") as file:
        data2 = json.load(file)
print("Number of entries:", len(data2))

for entry in data2:
    entry['response'] = resp[entry['context']]

print(data2)

Number of entries: 31
[{'query': 'What are my next classes?', 'current_time': '09:00 Monday', 'context': 'Lecture of PROBABILITY AND STATISTICS in Room 33-605 at 09:00 today.', 'response': 'You have a Probability and Statistics lecture in Room 33-605 at 09:00.'}, {'query': 'What are my next classes?', 'current_time': '10:00 Monday', 'context': 'Lecture of MATHEMATICS BEHIND MACHINE LEARNING in Room 33-605 at 10:00 today.', 'response': 'Your Mathematics Behind Machine Learning lecture is in Room 33-605 at 10:00.'}, {'query': 'What are my next classes?', 'current_time': '11:00 Monday', 'context': 'Lecture of MATHEMATICS BEHIND MACHINE LEARNING in Room 33-605 at 11:00 today.', 'response': 'You have another Mathematics Behind Machine Learning lecture in Room 33-605 at 11:00.'}, {'query': 'What are my next classes?', 'current_time': '12:00 Monday', 'context': 'Lecture of COMPUTER ORGANIZATION AND DESIGN in Room 33-605 at 12:00 today.', 'response': 'Your Computer Organization and Design lect

In [102]:
with open('timeTableContext.json', mode='w') as f:
        f.write(json.dumps(data2, indent=2))

In [100]:
a = []
with open('timeTableContext.json', mode='w') as f:
        f.write(json.dumps(a, indent=2))

with open('timeTableContext.json') as feedsjson:
    feeds = json.load(feedsjson)
    for data, time in formatted_data.items():
        timeTableData = {}
        timeTableData['query'] = "What are my next classes?"
        timeTableData['current_time'] = time
        timeTableData['context'] = data
        timeTableData['response'] = ""
        feeds.append(timeTableData)

with open('timeTableContext.json', mode='w') as f:
        f.write(json.dumps(feeds, indent=2))


In [50]:
a = []
with open('timeTableContext2.json', mode='w') as f:
        f.write(json.dumps(a, indent=2))

with open('timeTableContext2.json') as feedsjson:
    feeds = json.load(feedsjson)
    for data, time in formatted_data.items():
        timeTableData = {}
        timeTableData['query'] = "What are my next classes?"
        timeTableData['current_time'] = time
        timeTableData['context'] = data
        timeTableData['response'] = ""
        feeds.append(timeTableData)

with open('timeTableContext2.json', mode='w') as f:
        f.write(json.dumps(feeds, indent=2))
